In [ ]:
!pip install pysam

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import statistics
import matplotlib.pyplot as plt
from scipy.stats import binomtest
import statsmodels.formula.api as smf
import statsmodels.api as sm
from inference import COSMIC, EM
import pysam
import pyspark
import dxpy
import dxdata

In [ ]:
macs = {}

for chrom in tqdm(range(1, 23)):

    path = f'/mnt/project/DNM/sibs/diffs/chr{chrom}/'

    with open(f'{path}sibs_chr{chrom}.txt') as f:

        for row in f:

            row = row.strip()
            row = row.split(' ')

            diff_id = row[0]
            diff_mac = int(row[1])
            diff_obs = int(row[2])

            macs[diff_id] = (diff_mac, diff_obs)

In [ ]:
!dx download 'Bulk/Exome sequences/Exome OQFE CRAM files/helper_files/GRCh38_full_analysis_set_plus_decoy_hla.fa'
!dx download 'Bulk/Exome sequences/Exome OQFE CRAM files/helper_files/GRCh38_full_analysis_set_plus_decoy_hla.fa.fai'
fasta = pysam.FastaFile('GRCh38_full_analysis_set_plus_decoy_hla.fa')

## QC

In [ ]:
dp_lower_thre = 20
dp_upper_thre = 100

gq_thre = 30
p_thre = 0.05

In [ ]:
inds_diffs_qc = {}

path = f'/mnt/project/DNM/sibs/QC/'

for b in tqdm(range(224)):
    
    with open(f'{path}sibs_qc_b{b}.txt') as f:

        for row in f:
            
            row = row.strip()
            row = row.split(' ')

            ind_idx = str(row[0])
            if ind_idx not in inds_diffs_qc:
                inds_diffs_qc[ind_idx] = []

            chrom = int(row[1].split(':')[0])
            pos = int(row[1].split(':')[1])
            ref = row[1].split(':')[2]
            alt = row[1].split(':')[3]
                
            mac, obs = macs[row[1]]

            ref_ad = int(row[2])
            alt_ad = int(row[3])
            dp = int(row[4])
            gq = int(row[5])

            sib_ref_ad = int(row[8])
            sib_alt_ad = int(row[9])
            sib_dp = int(row[10])
            sib_gq = int(row[11])

            if (dp >= dp_lower_thre and sib_dp >= dp_lower_thre and
                gq >= gq_thre and sib_gq >= gq_thre and
                dp <= dp_upper_thre and sib_alt_ad == 0):

                p = binomtest(alt_ad, dp, 0.5).pvalue

                if p > p_thre:
                    inds_diffs_qc[ind_idx].append((chrom, pos, ref, alt,
                                                   mac, obs, alt_ad, dp, gq,
                                                   sib_alt_ad, sib_dp, sib_gq))

In [ ]:
with open('sibs_diffs_info_v0.txt', 'w') as f:
    f.write('IID\tCHROM\tPOS\tref\talt\tMAC\tOBS\n')
    for ind in inds_diffs_qc:
        for diff in inds_diffs_qc[ind]:
            f.write(f'{ind}\t{diff[0]}\t{diff[1]}\t{diff[2]}\t{diff[3]}\t{diff[4]}\t{diff[5]}\n')

In [ ]:
!dx upload sibs_diffs_info_v0.txt --path='DNM/sibs/out/'

In [ ]:
# inds_diffs_qc_df = pd.read_csv('/mnt/project/DNM/sibs/out/sibs_diffs_info_v0.txt', sep = '\t')

# inds_diffs_qc = {}

# for _, row in inds_diffs_qc_df.iterrows():
#     if row['IID'] not in inds_diffs_qc:
#         inds_diffs_qc[row['IID']] = []
#     inds_diffs_qc[row['IID']].append((row['CHROM'], row['POS'], row['ref'], 
#                                       row['alt'], row['MAC'], row['OBS']))

In [ ]:
cnts = [len(inds_diffs_qc[ind_idx]) for ind_idx in inds_diffs_qc]
                
print(min(cnts), max(cnts), round(statistics.mean(cnts), 2), statistics.median(cnts), sum(cnts), len(cnts))

In [ ]:
plt.boxplot(cnts)
plt.ylabel('Number of SNP differences per sib')
plt.show()

In [ ]:
plt.hist(cnts, bins = 20)
plt.xlabel('Number of SNP differences per sib')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

In [ ]:
cnts = []

with open('/mnt/project/DNM/sibs_pairs.txt') as f:
    
    for i, row in enumerate(f):
        
        row = row.strip()
        row = row.split(' ')
    
        sib1 = str(row[0])
        sib2 = str(row[1])
        
        b = int(i/100)
        p = i%100
            
        sib1_idx = f'{sib1}_b{b}_p{p}'
        sib2_idx = f'{sib2}_b{b}_p{p}'
        
        if sib1_idx not in inds_diffs_qc or sib2_idx not in inds_diffs_qc:
            continue
            
        cnts.append(len(inds_diffs_qc[sib1_idx]) + len(inds_diffs_qc[sib2_idx]))
        
print(min(cnts), max(cnts), round(statistics.mean(cnts), 2), statistics.median(cnts), sum(cnts), len(cnts))

In [ ]:
plt.boxplot(cnts)
plt.ylabel('Number of SNP differences per sib pair')
plt.show()

In [ ]:
plt.hist(cnts, bins = 20)
plt.xlabel('Number of SNP differences per sib pair')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

# Trim IBD2 segments

In [ ]:
ibd2 = {}

with open('/mnt/project/DNM/snipar/sibs_ibd2_0.75cM_trim_0cM_6cM_exc.bed') as f:
    
    for row in f:
        
        row = row.strip()
        
        if '>start:' in row:
            row = row.split(':')[1]
            sib1 = str(row.split('-')[0])
            sib2 = str(row.split('-')[1])
            ibd2[(sib1, sib2)] = {}
            continue
            
        if '>end:' in row:
            continue
            
        row = row.split(' ')
        chrom = int(row[0])
        start = int(row[1])
        end = int(row[2])-1
        
        if chrom not in ibd2[(sib1, sib2)]:
            ibd2[(sib1, sib2)][chrom] = []
        ibd2[(sib1, sib2)][chrom].append((start, end))

In [ ]:
inds_diffs_trim = {}

with open('/mnt/project/DNM/sibs_pairs.txt') as f:
    
    for i, row in tqdm(enumerate(f)):
        
        row = row.strip()
        row = row.split(' ')
    
        sib1 = str(row[0])
        sib2 = str(row[1])
        
        b = int(i/100)
        p = i%100
            
        sib1_idx = f'{sib1}_b{b}_p{p}'
        sib2_idx = f'{sib2}_b{b}_p{p}'
        
        if sib1_idx not in inds_diffs_qc or sib2_idx not in inds_diffs_qc:
            continue
        
        inds_diffs_trim[sib1_idx] = []
        inds_diffs_trim[sib2_idx] = []
            
        for diff in inds_diffs_qc[sib1_idx]:
        
            chrom = int(diff[0])
            pos = int(diff[1])
            if chrom not in ibd2[(sib1, sib2)]:
                continue

            inIBD2 = False
            for start, end in ibd2[(sib1, sib2)][chrom]:
                if pos >= start and pos <= end:
                    inIBD2 = True
                    break
            if inIBD2 == True:
                inds_diffs_trim[sib1_idx].append(diff)
                
        for diff in inds_diffs_qc[sib2_idx]:
        
            chrom = int(diff[0])
            pos = int(diff[1])
            if chrom not in ibd2[(sib1, sib2)]:
                continue

            inIBD2 = False
            for start, end in ibd2[(sib1, sib2)][chrom]:
                if pos >= start and pos <= end:
                    inIBD2 = True
                    break
            if inIBD2 == True:
                inds_diffs_trim[sib2_idx].append(diff)

In [ ]:
cnts = [len(inds_diffs_trim[ind_idx]) for ind_idx in inds_diffs_trim]
                
print(min(cnts), max(cnts), round(statistics.mean(cnts), 2), statistics.median(cnts), sum(cnts), len(cnts))

In [ ]:
plt.boxplot(cnts)
plt.ylabel('Number of SNP differences per sib')
plt.show()

In [ ]:
plt.hist(cnts, bins = 20)
plt.xlabel('Number of SNP differences per sib')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

In [ ]:
cnts = []

with open('/mnt/project/DNM/sibs_pairs.txt') as f:
    
    for i, row in enumerate(f):
        
        row = row.strip()
        row = row.split(' ')
    
        sib1 = str(row[0])
        sib2 = str(row[1])
        
        b = int(i/100)
        p = i%100
            
        sib1_idx = f'{sib1}_b{b}_p{p}'
        sib2_idx = f'{sib2}_b{b}_p{p}'
        
        if sib1_idx not in inds_diffs_trim or sib2_idx not in inds_diffs_trim:
            continue
            
        cnts.append(len(inds_diffs_trim[sib1_idx]) + len(inds_diffs_trim[sib2_idx]))
        
print(min(cnts), max(cnts), round(statistics.mean(cnts), 2), statistics.median(cnts), sum(cnts), len(cnts))

In [ ]:
plt.boxplot(cnts)
plt.ylabel('Number of SNP differences per sib pair')
plt.show()

In [ ]:
plt.hist(cnts, bins = 20)
plt.xlabel('Number of SNP differences per sib pair')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

# Exclude clusters

In [ ]:
def exclude_clusters(ind_diffs, window = 100000, thre = 3):
    
    ind_diffs = sorted(ind_diffs, key = lambda x: (x[0], x[1]))
    
    ind_diffs_chrom = {chrom: [] for chrom in range(1, 23)}
    for diff in ind_diffs:
        ind_diffs_chrom[diff[0]].append(diff)
        
    ind_diffs_exc = []
    
    for chrom in ind_diffs_chrom:
        
        pointer = 0
    
        for i in range(len(ind_diffs_chrom[chrom])):

            while (ind_diffs_chrom[chrom][i][0] == ind_diffs_chrom[chrom][pointer][0] and 
                   (ind_diffs_chrom[chrom][i][1]-ind_diffs_chrom[chrom][pointer][1]) > window):
                pointer += 1

            if (i-pointer+1) >= thre:
                for j in range(pointer, i+1):
                    ind_diffs_exc.append(ind_diffs_chrom[chrom][j])

    return [diff for diff in ind_diffs if diff not in ind_diffs_exc]

In [ ]:
inds_diffs_cluster = {ind_idx: exclude_clusters(inds_diffs_trim[ind_idx]) for ind_idx in inds_diffs_trim}

In [ ]:
cnts = [len(inds_diffs_cluster[ind_idx]) for ind_idx in inds_diffs_cluster]
                
print(min(cnts), max(cnts), round(statistics.mean(cnts), 2), statistics.median(cnts), sum(cnts), len(cnts))

In [ ]:
plt.boxplot(cnts)
plt.ylabel('Number of SNP differences per sib')
plt.show()

In [ ]:
plt.hist(cnts, bins = 20)
plt.xlabel('Number of SNP differences per sib')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

In [ ]:
cnts = []

with open('/mnt/project/DNM/sibs_pairs.txt') as f:
    
    for i, row in enumerate(f):
        
        row = row.strip()
        row = row.split(' ')
    
        sib1 = str(row[0])
        sib2 = str(row[1])
        
        b = int(i/100)
        p = i%100
            
        sib1_idx = f'{sib1}_b{b}_p{p}'
        sib2_idx = f'{sib2}_b{b}_p{p}'
        
        if sib1_idx not in inds_diffs_cluster or sib2_idx not in inds_diffs_cluster:
            continue
            
        cnts.append(len(inds_diffs_cluster[sib1_idx]) + len(inds_diffs_cluster[sib2_idx]))
        
print(min(cnts), max(cnts), round(statistics.mean(cnts), 2), statistics.median(cnts), sum(cnts), len(cnts))

In [ ]:
plt.boxplot(cnts)
plt.ylabel('Number of SNP differences per sib pair')
plt.show()

In [ ]:
plt.hist(cnts, bins = 20)
plt.xlabel('Number of SNP differences per sib pair')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

# Exclude common variants

In [ ]:
def exclude_common(ind_diffs, maf = 0.001):
    
    return [diff for diff in ind_diffs if float(diff[4])/float(diff[5]) <= maf]

In [ ]:
inds_diffs_maf = {ind_idx: exclude_common(inds_diffs_cluster[ind_idx]) for ind_idx in inds_diffs_cluster}

In [ ]:
cnts = [len(inds_diffs_maf[ind_idx]) for ind_idx in inds_diffs_maf]
                
print(min(cnts), max(cnts), round(statistics.mean(cnts), 2), statistics.median(cnts), sum(cnts), len(cnts))

In [ ]:
plt.boxplot(cnts)
plt.ylabel('Number of SNP differences per sib')
plt.show()

In [ ]:
plt.hist(cnts, bins = 20)
plt.xlabel('Number of SNP differences per sib')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

In [ ]:
cnts = []

with open('/mnt/project/DNM/sibs_pairs.txt') as f:
    
    for i, row in enumerate(f):
        
        row = row.strip()
        row = row.split(' ')
    
        sib1 = str(row[0])
        sib2 = str(row[1])
        
        b = int(i/100)
        p = i%100
            
        sib1_idx = f'{sib1}_b{b}_p{p}'
        sib2_idx = f'{sib2}_b{b}_p{p}'
        
        if sib1_idx not in inds_diffs_maf or sib2_idx not in inds_diffs_maf:
            continue
            
        cnts.append(len(inds_diffs_maf[sib1_idx]) + len(inds_diffs_maf[sib2_idx]))
        
print(min(cnts), max(cnts), round(statistics.mean(cnts), 2), statistics.median(cnts), sum(cnts), len(cnts))

In [ ]:
plt.boxplot(cnts)
plt.ylabel('Number of SNP differences per sib pair')
plt.show()

In [ ]:
plt.hist(cnts, bins = 20)
plt.xlabel('Number of SNP differences per sib pair')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

# Exclude outliers

In [ ]:
# Clinical data
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)

dispensed_database_name = dxpy.find_one_data_object(classname = 'database', name = 'app*', folder = '/', name_mode = 'glob', describe = True)['describe']['name']
dispensed_dataset_id = dxpy.find_one_data_object(typename = 'Dataset', name = 'app*.dataset', folder = '/', name_mode = 'glob')['id']

dataset = dxdata.load_dataset(id = dispensed_dataset_id)
participant = dataset['participant']

field_names = ['eid', 'p34']

df = participant.retrieve_fields(names = field_names, engine = dxdata.connect())

inds_sql_df = df.toPandas()
inds_sql_df['eid'] = inds_sql_df['eid'].astype(int)
inds_sql_df = inds_sql_df.dropna(subset = ['p34'])
inds_sql_df.head()

In [ ]:
ind_yob = inds_sql_df.set_index('eid')['p34'].to_dict()

In [ ]:
data = []

with open('/mnt/project/DNM/sibs_pairs.txt') as f:
    
    for i, row in enumerate(f):
        
        row = row.strip()
        row = row.split(' ')
    
        sib1 = int(row[0])
        sib2 = int(row[1])
        
        b = int(i/100)
        p = i%100
            
        sib1_idx = f'{sib1}_b{b}_p{p}'
        sib2_idx = f'{sib2}_b{b}_p{p}'
        
        if sib1_idx not in inds_diffs_maf or sib2_idx not in inds_diffs_maf:
            continue
        
        older = sib1 if ind_yob[sib1] <= ind_yob[sib2] else sib2
        younger = sib1 if ind_yob[sib1] > ind_yob[sib2] else sib2
        
        older_idx = f'{older}_b{b}_p{p}'
        younger_idx = f'{younger}_b{b}_p{p}'
        
        data.append((older_idx, younger_idx, ind_yob[younger]-ind_yob[older],
                     len(inds_diffs_maf[older_idx]), len(inds_diffs_maf[younger_idx]),
                     len(inds_diffs_maf[older_idx])-len(inds_diffs_maf[younger_idx])))
        
data = sorted(data, key = lambda x: x[2])

In [ ]:
data_df = pd.DataFrame(data, columns = ['older_idx', 'younger_idx', 'age_diff', 'older_dnm', 'younger_dnm', 'dnm_diff'])

model = smf.glm(formula = 'dnm_diff ~ age_diff', data = data_df).fit()
model.summary()

In [ ]:
preds = (model.params['Intercept'] +
         model.params['age_diff'] * data_df['age_diff'])

ci = model.get_prediction(data_df).summary_frame(alpha = 0.05)

sorted_idx = np.argsort(data_df['age_diff'])
age_diff_sorted = data_df['age_diff'].values[sorted_idx]
preds_sorted = preds.values[sorted_idx]
ci_lower_sorted = ci['mean_ci_lower'].values[sorted_idx]
ci_upper_sorted = ci['mean_ci_upper'].values[sorted_idx]

plt.fill_between(age_diff_sorted, ci_lower_sorted, ci_upper_sorted, 
                 color = 'lightgrey', alpha = 0.5, label = '95% CI')

plt.scatter(data_df['age_diff'], data_df['dnm_diff'], alpha = 0.7)
plt.plot(age_diff_sorted, preds_sorted)

plt.xlabel('Age difference')
plt.ylabel('De novo SNP count difference')

plt.grid(True)
plt.legend()
plt.show()

In [ ]:
stats_df = data_df.groupby('age_diff').apply(lambda df: pd.Series({
    'mean_diff': df['dnm_diff'].mean(),
    'std': np.sqrt(df['older_dnm'].mean()+df['younger_dnm'].mean())
}), include_groups = False).reset_index()

data_df = data_df.merge(stats_df, on = 'age_diff', how = 'left')
data_df['out'] = (abs(data_df['dnm_diff']-data_df['mean_diff']) > 3*data_df['std'])

exc = list(data_df.loc[data_df['out'], ['older_idx', 'younger_idx']].itertuples(index = False, name = None))

In [ ]:
data_qc = [info for info in data if (info[0], info[1]) not in exc]

In [ ]:
data_qc_df = pd.DataFrame(data_qc, columns = ['older_idx', 'younger_idx', 'age_diff', 'older_dnm', 'younger_dnm', 'dnm_diff'])

model = smf.glm(formula = 'dnm_diff ~ age_diff', data = data_qc_df).fit()
model.summary()

In [ ]:
preds = (model.params['Intercept'] +
         model.params['age_diff'] * data_qc_df['age_diff'])

ci = model.get_prediction(data_qc_df).summary_frame(alpha = 0.05)

sorted_idx = np.argsort(data_qc_df['age_diff'])
age_diff_sorted = data_qc_df['age_diff'].values[sorted_idx]
preds_sorted = preds.values[sorted_idx]
ci_lower_sorted = ci['mean_ci_lower'].values[sorted_idx]
ci_upper_sorted = ci['mean_ci_upper'].values[sorted_idx]

plt.fill_between(age_diff_sorted, ci_lower_sorted, ci_upper_sorted, 
                 color = 'lightgrey', alpha = 0.5, label = '95% CI')

plt.scatter(data_qc_df['age_diff'], data_qc_df['dnm_diff'], alpha = 0.7)
plt.plot(age_diff_sorted, preds_sorted)

plt.xlabel('Age difference')
plt.ylabel('De novo SNP count difference')

plt.grid(True)
plt.legend()
plt.show()

In [ ]:
inds_exc = [ind for inds in exc for ind in inds]
inds_diffs_out = {ind_idx: inds_diffs_maf[ind_idx] for ind_idx in inds_diffs_maf if ind_idx not in inds_exc}

In [ ]:
cnts = [len(inds_diffs_out[ind_idx]) for ind_idx in inds_diffs_out]
                
print(min(cnts), max(cnts), round(statistics.mean(cnts), 2), statistics.median(cnts), sum(cnts), len(cnts))

In [ ]:
plt.boxplot(cnts)
plt.ylabel('Number of SNP differences per sib')
plt.show()

In [ ]:
plt.hist(cnts, bins = 20)
plt.xlabel('Number of SNP differences per sib')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

In [ ]:
cnts = []

with open('/mnt/project/DNM/sibs_pairs.txt') as f:
    
    for i, row in enumerate(f):
        
        row = row.strip()
        row = row.split(' ')
    
        sib1 = str(row[0])
        sib2 = str(row[1])
        
        b = int(i/100)
        p = i%100
            
        sib1_idx = f'{sib1}_b{b}_p{p}'
        sib2_idx = f'{sib2}_b{b}_p{p}'
        
        if sib1_idx in inds_diffs_out and sib2_idx in inds_diffs_out:
            cnts.append(len(inds_diffs_out[sib1_idx]) + len(inds_diffs_out[sib2_idx]))
        
print(min(cnts), max(cnts), round(statistics.mean(cnts), 2), statistics.median(cnts), sum(cnts), len(cnts))

In [ ]:
plt.boxplot(cnts)
plt.ylabel('Number of SNP differences per sib pair')
plt.show()

In [ ]:
plt.hist(cnts, bins = 20)
plt.xlabel('Number of SNP differences per sib pair')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

# Gene conversion events

In [ ]:
dmc1 = {}

dmc1_df = pd.read_csv('/mnt/project/DNM/dmc1_hotspots.csv')

for _, row in dmc1_df.iterrows():
    
    if (row['Chromosome'] == 'chrX' or 
        row['Chromosome'] == 'chrY'):
        continue
    
    chrom = int(row['Chromosome'][3:])
    start = int(row['Start_Pos'])
    end = int(row['End_Pos'])
    
    if chrom not in dmc1:
        dmc1[chrom] = []
    dmc1[chrom].append((start, end))

In [ ]:
cnt = 0
total = 0

for ind in inds_diffs_cluster:
    
    for diff in inds_diffs_cluster[ind]:
        
        in_dmc1 = False
        
        for start, end in dmc1[diff[0]]:
            if diff[1] >= start and diff[1] <= end:
                in_dmc1 = True
                
        if in_dmc1 == True:
            cnt += 1
        total += 1
        
print(cnt, total, round(cnt/total, 4))

In [ ]:
cnt = 0
total = 0

for ind in inds_diffs_maf:
    
    for diff in inds_diffs_maf[ind]:
        
        in_dmc1 = False
        
        for start, end in dmc1[diff[0]]:
            if diff[1] >= start and diff[1] <= end:
                in_dmc1 = True
                
        if in_dmc1 == True:
            cnt += 1
        total += 1
        
print(cnt, total, round(cnt/total, 4))

In [ ]:
cnt = 0
total = 0

for ind in inds_diffs_out:
    
    for diff in inds_diffs_out[ind]:
        
        in_dmc1 = False
        
        for start, end in dmc1[diff[0]]:
            if diff[1] >= start and diff[1] <= end:
                in_dmc1 = True
                
        if in_dmc1 == True:
            cnt += 1
        total += 1
        
print(cnt, total, round(cnt/total, 4))

# Mutational spectrum and signatures

In [ ]:
def get_types(diffs):

    muts = {}

    for ref in ['C', 'T']:
        for alt in ['A', 'C', 'G', 'T']:
            if ref != alt:
                for prev in ['A', 'C', 'G', 'T']:
                    for next in ['A', 'C', 'G', 'T']:
                        muts[f'{prev}[{ref}>{alt}]{next}'] = 0

    comp = {'A':'T', 'T':'A', 'G':'C', 'C':'G'}

    for diff in diffs:

        chrom = diff[0]
        pos = diff[1]
        ref = diff[2]
        alt = diff[3]

        cont = fasta.fetch(f'chr{chrom}', pos-2, pos+1)
        mut = f'{cont[0]}[{ref}>{alt}]{cont[2]}'

        if ref == 'A' or ref == 'G':
            ref = comp[ref]
            alt = comp[alt]
            cont = f'{comp[cont[0]]}{comp[cont[1]]}{comp[cont[2]]}'
            cont = cont[::-1]
            mut = (comp[mut[6]]+mut[1]+comp[mut[2]]+mut[3]+ 
                   comp[mut[4]]+mut[5]+comp[mut[0]])

        muts[mut] += 1

    return muts

In [ ]:
def merge_types(inds_types):
    
    muts = {}

    for ref in ['C', 'T']:
        for alt in ['A', 'C', 'G', 'T']:
            if ref != alt:
                for prev in ['A', 'C', 'G', 'T']:
                    for next in ['A', 'C', 'G', 'T']:
                        muts[f'{prev}[{ref}>{alt}]{next}'] = 0
                        
    for ind in inds_types:
        for mut in inds_types[ind]:
            muts[mut] += inds_types[ind][mut]
            
    muts = {mut: muts[mut]/sum(muts.values()) for mut in muts}
            
    return muts

In [ ]:
def list_types(inds_types):
    
    muts = []
    
    for ind in inds_types:
        for mut in inds_types[ind]:
            for i in range(inds_types[ind][mut]):
                muts.append(mut)
            
    muts = pd.DataFrame({'Type': muts})
    
    return muts

In [ ]:
def plot_spectrum(spectrum):

    colors = {
        'C>A': '#1f77b4',
        'C>G': '#000000',
        'C>T': '#e41a1c',
        'T>A': '#999999',
        'T>C': '#4daf4a',
        'T>G': '#ff7f00'
    }

    types = list(spectrum.keys())
    fracts = list(spectrum.values())
    groups = [colors[group[2:5]] for group in spectrum]

    plt.figure(figsize = (20, 5))
    plt.bar(types, fracts, color = groups, width = 0.8)
    plt.xticks(rotation = 90, fontsize = 7)

    plt.ylabel('Fraction')
    plt.xlabel('Mutation type')
    plt.show()
    plt.clf()

In [ ]:
def plot_signatures(spectrum):
    
    em = EM(spectrum, COSMIC(), beta = 0)
    em.infer()
    em.plot(first = 5)
    return em.fracts()

In [ ]:
inds_types = {ind: get_types(inds_diffs_out[ind]) for ind in inds_diffs_out}

In [ ]:
spectrum = merge_types(inds_types)
plot_spectrum(spectrum)

In [ ]:
spectrum = list_types(inds_types)
print(plot_signatures(spectrum))

# Record differences

In [ ]:
with open('sibs_diffs_info_updated.txt', 'w') as f:
    f.write('IID\tCHROM\tPOS\tref\talt\tMAC\tOBS\n')
    for ind in inds_diffs_out:
        for diff in inds_diffs_out[ind]:
            f.write(f'{ind}\t{diff[0]}\t{diff[1]}\t{diff[2]}\t{diff[3]}\t{diff[4]}\t{diff[5]}\n')

In [ ]:
spectrum = merge_types(inds_types)
with open('sibs_diffs_types_updated.txt', 'w') as f:
    f.write('Type\tFraction\n')
    for type in spectrum:
        f.write(f'{type}\t{spectrum[type]}\n')

In [ ]:
with open('sibs_diffs_sigs_updated.txt', 'w') as f:
    f.write('Project\tSample\tID\tGenome\tmut_type\tchrom\tpos_start\tpos_end\tref\talt\tType\n')
    for idx, ind in enumerate(inds_diffs_out):
        for diff in inds_diffs_out[ind]:
            f.write(f'.\t{idx}\t.\tGRCh38\tSNP\t{diff[0]}\t{diff[1]}\t{diff[1]}\t{diff[2]}\t{diff[3]}\t.\n')

In [ ]:
with open('sibs_diffs_vep_updated.vcf', 'w') as f:
    for ind in inds_diffs_out:
        for diff in inds_diffs_out[ind]:
            f.write(f'{diff[0]} {diff[1]} {diff[2]} {diff[3]}\n')

In [ ]:
!dx upload 'sibs_diffs_info_updated.txt' --path='DNM/sibs/out/'
!dx upload 'sibs_diffs_types_updated.txt' --path='DNM/sibs/out/'
!dx upload 'sibs_diffs_sigs_updated.txt' --path='DNM/sibs/out/'
!dx upload 'sibs_diffs_vep_updated.vcf' --path='DNM/sibs/out/'